# Session 08 Practice: Conformal Lattice Planning in Frenet Frame
**Course:** Motion Planning for Autonomous Vehicles (Planning P2)  
**Week 4 | Session 8**  

---

### 🎯 Learning Objectives
1. Implement **Quintic Polynomial** parametric curves satisfying 1D boundary conditions $(d_0, \dot{d}_0, \ddot{d}_0) \to (d_f, \dot{d}_f, \ddot{d}_f)$.
2. Construct road-aligned **Conformal Spatiotemporal Lattices** in Frenet coordinates $(s, d)$.
3. Transform paths from Frenet frame $(s, d)$ to global Cartesian $(x, y)$ along curved reference centerlines.
4. Visualize lattice bundles, curvature profiles, and lateral jerk using **Plotly**.

---

### 📐 Mathematical Formulation

#### Quintic Polynomial in Frenet Space
$$d(s) = a_0 + a_1 s + a_2 s^2 + a_3 s^3 + a_4 s^4 + a_5 s^5$$

Boundary conditions at $s = 0$ and $s = S_f$:
$$
egin{bmatrix}
S_f^3 & S_f^4 & S_f^5 \
3S_f^2 & 4S_f^3 & 5S_f^4 \
6S_f & 12S_f^2 & 20S_f^3
\end{bmatrix}
egin{bmatrix}
a_3 \ a_4 \ a_5
\end{bmatrix}
=
egin{bmatrix}
d_f - a_0 - a_1 S_f - a_2 S_f^2 \
\dot{d}_f - a_1 - 2 a_2 S_f \
\ddot{d}_f - 2 a_2
\end{bmatrix}
$$

#### Frenet to Cartesian Transformation
$$\mathbf{r}(s, d) = \mathbf{r}_{	ext{ref}}(s) + d \mathbf{n}(s) = egin{bmatrix} x_r(s) - d \sin	heta_r(s) \ y_r(s) + d \cos	heta_r(s) \end{bmatrix}$$


In [ ]:
import math
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "plotly_white"


## 1. Quintic Polynomial & Coordinate Transform Formulation

In [ ]:
class QuinticPolynomial1D:
    def __init__(self, xs: float, vxs: float, axs: float,
                 xe: float, vxe: float, axe: float, T: float):
        self.a0 = xs
        self.a1 = vxs
        self.a2 = axs / 2.0
        
        A = np.array([
            [T**3, T**4, T**5],
            [3*T**2, 4*T**3, 5*T**4],
            [6*T, 12*T**2, 20*T**3]
        ])
        b = np.array([
            xe - self.a0 - self.a1*T - self.a2*T**2,
            vxe - self.a1 - 2*self.a2*T,
            axe - 2*self.a2
        ])
        
        coeffs = np.linalg.solve(A, b)
        self.a3 = coeffs[0]
        self.a4 = coeffs[1]
        self.a5 = coeffs[2]

    def calc_point(self, t: float) -> float:
        return self.a0 + self.a1*t + self.a2*t**2 + self.a3*t**3 + self.a4*t**4 + self.a5*t**5

    def calc_first_derivative(self, t: float) -> float:
        return self.a1 + 2*self.a2*t + 3*self.a3*t**2 + 4*self.a4*t**3 + 5*self.a5*t**4

    def calc_second_derivative(self, t: float) -> float:
        return 2*self.a2 + 6*self.a3*t + 12*self.a4*t**2 + 20*self.a5*t**3

def frenet_to_cartesian(s_vals: np.ndarray, d_vals: np.ndarray, ref_line_fn) -> list[tuple[float, float]]:
    cartesian_pts = []
    for s, d in zip(s_vals, d_vals):
        xr, yr, thetar = ref_line_fn(s)
        x = xr - d * math.sin(thetar)
        y = yr + d * math.cos(thetar)
        cartesian_pts.append((x, y))
    return cartesian_pts


## 2. Conformal Spatiotemporal Lattice Generator

In [ ]:
def generate_conformal_lattice(s0: float = 0.0, d0: float = 0.0,
                               longitudinal_horizons: list[float] = [25.0, 35.0, 45.0],
                               lateral_offsets: list[float] = [-3.5, 0.0, 3.5],
                               ds: float = 0.5):
    lattice_paths = []
    records = []
    
    # Curved reference line with radius R = 180m
    def curved_reference(s):
        R = 180.0
        angle = s / R
        xr = R * math.sin(angle)
        yr = R * (1.0 - math.cos(angle))
        thetar = angle
        return xr, yr, thetar
        
    path_id = 0
    for s_target in longitudinal_horizons:
        for d_target in lateral_offsets:
            poly = QuinticPolynomial1D(xs=d0, vxs=0.0, axs=0.0,
                                       xe=d_target, vxe=0.0, axe=0.0,
                                       T=s_target)
            
            s_samples = np.arange(0.0, s_target + ds, ds)
            d_samples = np.array([poly.calc_point(s) for s in s_samples])
            d_prime = np.array([poly.calc_first_derivative(s) for s in s_samples])
            d_double_prime = np.array([poly.calc_second_derivative(s) for s in s_samples])
            
            max_lat_curvature = float(np.max(np.abs(d_double_prime)))
            pts = frenet_to_cartesian(s0 + s_samples, d_samples, curved_reference)
            
            lattice_paths.append({
                "path_id": path_id,
                "s_target": s_target,
                "d_target": d_target,
                "s_samples": s_samples,
                "d_samples": d_samples,
                "d_double_prime": d_double_prime,
                "points": pts,
                "max_lat_curv": max_lat_curvature
            })
            
            records.append({
                "path_id": path_id,
                "s_target_m": s_target,
                "d_target_m": d_target,
                "num_waypoints": len(pts),
                "max_lateral_curvature": round(max_lat_curvature, 4)
            })
            path_id += 1
            
    return lattice_paths, pl.DataFrame(records), curved_reference

lattice_paths, df_meta, ref_fn = generate_conformal_lattice()
print(f"Generated {len(lattice_paths)} conformal lattice trajectory candidates.")


## 3. Interactive Road-Aligned Lattice Visualizer

In [ ]:
# 1. Global Cartesian Road & Lattice Bundle
fig_lat = go.Figure()

# Road Reference Line
s_ref = np.linspace(0, 50, 150)
xr_ref = [ref_fn(s)[0] for s in s_ref]
yr_ref = [ref_fn(s)[1] for s in s_ref]
fig_lat.add_trace(go.Scatter(
    x=xr_ref, y=yr_ref, mode="lines",
    line=dict(color="black", dash="dash", width=3),
    name="Road Centerline (Reference)"
))

# Left & Right Lane Boundaries (offset by +- 3.5m)
xr_left = [ref_fn(s)[0] - 3.5 * math.sin(ref_fn(s)[2]) for s in s_ref]
yr_left = [ref_fn(s)[1] + 3.5 * math.cos(ref_fn(s)[2]) for s in s_ref]
xr_right = [ref_fn(s)[0] + 3.5 * math.sin(ref_fn(s)[2]) for s in s_ref]
yr_right = [ref_fn(s)[1] - 3.5 * math.cos(ref_fn(s)[2]) for s in s_ref]

fig_lat.add_trace(go.Scatter(x=xr_left, y=yr_left, mode="lines", line=dict(color="#bbb", width=2), showlegend=False))
fig_lat.add_trace(go.Scatter(x=xr_right, y=yr_right, mode="lines", line=dict(color="#bbb", width=2), showlegend=False))

# Conformal Lattice Paths
for p in lattice_paths:
    xs = [pt[0] for pt in p["points"]]
    ys = [pt[1] for pt in p["points"]]
    fig_lat.add_trace(go.Scatter(
        x=xs, y=ys, mode="lines",
        line=dict(width=2),
        name=f"Lattice #{p['path_id']} (s={p['s_target']}m, d={p['d_target']}m)"
    ))

fig_lat.update_layout(
    title="Conformal Spatiotemporal Lattice Planning in Global Frame",
    xaxis_title="Global X (m)",
    yaxis_title="Global Y (m)",
    width=850,
    height=650
)
fig_lat.show()


In [ ]:
# 2. Frenet Space (s vs d) & Curvature Profiles
fig_frenet = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Frenet Frame Trajectories (s vs d)", "Lateral Curvature Profiles d''(s)")
)

for p in lattice_paths:
    fig_frenet.add_trace(
        go.Scatter(x=p["s_samples"], y=p["d_samples"], mode="lines", name=f"Path #{p['path_id']}"),
        row=1, col=1
    )
    fig_frenet.add_trace(
        go.Scatter(x=p["s_samples"], y=p["d_double_prime"], mode="lines", name=f"Curv #{p['path_id']}"),
        row=1, col=2
    )

fig_frenet.update_layout(
    title="Frenet Space Parametric Curves and Dynamic Curvature Analysis",
    height=450,
    showlegend=False
)
fig_frenet.show()


## 4. Verification Tests

In [ ]:
# Verify quintic boundary conditions for each candidate
for p in lattice_paths:
    d_start = p["d_samples"][0]
    d_end = p["d_samples"][-1]
    assert math.isclose(d_start, 0.0, abs_tol=1e-3), "Start lateral offset must be 0!"
    assert math.isclose(d_end, p["d_target"], abs_tol=1e-3), f"End offset did not match target {p['d_target']}!"

print("✅ Conformal Lattice boundary conditions verified successfully!")
